# 🛣️ ROADGUARD AI — HUẤN LUYỆN YOLOV11 MULTI-CLASS (ROADGUARD MULTICLASS V1)
### Mô Hình Đa Lớp: Ổ Gà (0: pothole) | Vết Nứt (1: crack) | Bong Tróc (2: bong_troc)
---
> **Mục tiêu:** Huấn luyện mô hình đa lớp hoàn chỉnh `roadguard_multiclass_v1` bằng cách fine-tune từ trọng số `weights/trained/roadguard_best_15k.pt` trên dataset hợp nhất `merged/` (~21,424 ảnh thực tế đã khử trùng lặp + 6,334 mẫu âm tính chống báo động giả).
> **Taxonomy chuẩn hóa (từ class_map.json):**
> - `0: pothole` (10,111 BBoxes)
> - `1: crack` (21,587 BBoxes)
> - `2: bong_troc` (Đang bổ sung dữ liệu)
> **Lệnh huấn luyện:** `yolo detect train data=merged/data.yaml model=weights/trained/roadguard_best_15k.pt epochs=120 imgsz=640 batch=16 patience=20 name=roadguard_multiclass_v1`
> **Phần cứng khuyến nghị:** Google Colab GPU Tesla T4 (Miễn phí) hoặc GPU RTX cá nhân.

## 📌 Bước 1: Kiểm Tra Trạng Thái GPU & Cài Đặt Ultralytics

In [ ]:
!nvidia-smi

import torch
print('=' * 65)
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ CẢNH BÁO: Chưa kích hoạt GPU! Vào Runtime -> Change runtime type -> Chọn T4 GPU')
print('=' * 65)

!pip install -q ultralytics pyyaml matplotlib

## 📥 Bước 2: Nạp Dataset Hợp Nhất & Trọng Số Nền Tảng
---
> Kéo thả file `merged.zip` và `roadguard_best_15k.pt` vào cột Files (📁) bên trái Colab, sau đó bấm chạy ô bên dưới.

In [ ]:
import os, glob, zipfile
from pathlib import Path

work_dir = Path('/content')
merged_dir = work_dir / 'merged'
merged_dir.mkdir(parents=True, exist_ok=True)

zip_files = glob.glob('/content/*.zip') + glob.glob('*.zip')
print(f'🔍 Tìm thấy {len(zip_files)} file zip: {zip_files}')

for z in zip_files:
    if 'merged' in z.lower():
        print(f'🚀 Đang giải nén {z} vào {merged_dir}...')
        with zipfile.ZipFile(z, 'r') as zf:
            zf.extractall(merged_dir)
        break

yaml_path = merged_dir / 'data.yaml'
if not yaml_path.exists():
    # Tạo data.yaml nếu chưa có
    cfg = {
        'path': str(merged_dir.resolve()),
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'nc': 3,
        'names': {0: 'pothole', 1: 'crack', 2: 'bong_troc'}
    }
    import yaml
    with open(yaml_path, 'w', encoding='utf-8') as f:
        yaml.dump(cfg, f, default_flow_style=False)

print('✅ data.yaml đã sẵn sàng tại:', yaml_path)
with open(yaml_path, 'r', encoding='utf-8') as f:
    print(f.read())

## 🚀 Bước 3: Huấn Luyện roadguard_multiclass_v1 (120 Epochs, Early Stopping 20)

In [ ]:
from ultralytics import YOLO

base_weight = 'roadguard_best_15k.pt' if Path('roadguard_best_15k.pt').exists() else 'yolo11n.pt'
print(f'🚀 Bắt đầu fine-tune từ mô hình nền tảng: {base_weight}')

model = YOLO(base_weight)

results = model.train(
    data=str(yaml_path),
    epochs=120,
    imgsz=640,
    batch=16,
    patience=20,
    name='roadguard_multiclass_v1',
    project='runs/detect',
    save=True,
    save_period=10,
    plots=True,
    device=0 if torch.cuda.is_available() else 'cpu'
)
print('🎉 TIẾN TRÌNH HUẤN LUYỆN 120 EPOCHS ĐÃ HOÀN TẤT!')

## 📊 Bước 4: Đánh Giá Chi Tiết Per-Class AP, Confusion Matrix & Xuất Mô Hình

In [ ]:
from IPython.display import Image, display

run_dir = Path('runs/detect/roadguard_multiclass_v1')
best_pt = run_dir / 'weights' / 'best.pt'

trained_model = YOLO(str(best_pt))
# Thiết lập project và name chuẩn xác để loại bỏ hoàn toàn lỗi lồng thư mục runs/detect/runs/detect/
metrics = trained_model.val(
    data=str(yaml_path),
    split='test',
    project='runs/detect',
    name='roadguard_multiclass_v1_eval',
    exist_ok=True,
    plots=True
)
eval_dir = Path(metrics.save_dir)

print('=' * 65)
print('🏆 KẾT QUẢ ĐÁNH GIÁ CHI TIẾT PER-CLASS AP:')
low_ap_classes = []
for i, name in trained_model.names.items():
    ap50 = metrics.box.ap50[i] * 100 if i < len(metrics.box.ap50) else 0.0
    ap = metrics.box.ap[i] * 100 if i < len(metrics.box.ap) else 0.0
    print(f' - Lớp {i} ({name:10s}): AP@50 = {ap50:.2f}% | AP@50-95 = {ap:.2f}%')
    if ap50 < 60.0:
        low_ap_classes.append((i, name, ap50))
print(f' - Toàn thể mAP@50   : {metrics.box.map50 * 100:.2f}%')
print(f' - Toàn thể mAP@50-95: {metrics.box.map * 100:.2f}%')
print('=' * 65)

if low_ap_classes:
    print('⚠️ CÁC LỚP CÓ ĐỘ CHÍNH XÁC AP < 0.6 CẦN BỔ SUNG DỮ LIỆU:')
    for cid, cname, cap in low_ap_classes:
        print(f'   👉 Lớp {cid} ({cname}): AP={cap:.2f}% < 60% -> Đề xuất bổ sung thêm ảnh thực địa!')

cm_p = eval_dir / 'confusion_matrix.png'
if cm_p.exists():
    display(Image(str(cm_p)))
elif (run_dir / 'confusion_matrix.png').exists():
    display(Image(str(run_dir / 'confusion_matrix.png')))